In [9]:
import numpy as np
import random
import joblib
import json
import time
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from deap import base, creator, tools, algorithms

import warnings
warnings.filterwarnings('ignore')

In [10]:
class CocktailTastePredictor(BaseEstimator, ClassifierMixin):
    def __init__(self, models_dict, thresholds, feature_columns, flavor_categories):
        self.models_dict = models_dict
        self.thresholds = thresholds
        self.feature_columns = feature_columns
        self.flavor_categories = flavor_categories
        
    def preprocess(self, input_data):
        if isinstance(input_data, np.ndarray):
            if input_data.shape[1] != len(self.feature_columns):
                raise ValueError(f"Expected {len(self.feature_columns)} columns, got {input_data.shape[1]}")
            return input_data

        if isinstance(input_data, list):
            return np.array([[item.get(col, 0.0) for col in self.feature_columns] for item in input_data])

        if isinstance(input_data, dict):
            return np.array([[input_data.get(col, 0.0) for col in self.feature_columns]])
            
        elif isinstance(input_data, pd.DataFrame):
            missing_cols = set(self.feature_columns) - set(input_data.columns)
            if missing_cols:
                input_data = pd.concat([input_data, pd.DataFrame(0, index=input_data.index, columns=list(missing_cols))], axis=1)
            return input_data[self.feature_columns].fillna(0.0).values
        else:
            raise ValueError("Input must be a numpy array, list, dict, or pandas DataFrame")

    def predict_proba(self, X):
        features = self.preprocess(X)
        probas = {}
        for flavor in self.flavor_categories:
            probas[flavor] = self.models_dict[flavor].predict_proba(features)[:, 1]
        return probas

    def predict(self, X):
        probas = self.predict_proba(X)
        predictions = {}
        for flavor in self.flavor_categories:
            predictions[flavor] = (probas[flavor] >= self.thresholds[flavor]).astype(int)
        return predictions

In [11]:
GA_CONFIG = {
    'POPULATION_SIZE': 1000,
    'GENERATIONS': 150,
    'ELITISM_COUNT': 10,
    'CROSSOVER_PROB': 0.7,
    'MUTATION_PROB': 0.3,
    'MUTATION_STRENGTH': 0.2,
    'TOURNAMENT_SIZE': 3,
    'SELECTION_PRESSURE': 0.7,
    'MIN_INGREDIENT_PCT': 0.01,
    'MAX_INGREDIENT_PCT': 0.35,
    'MIN_MANDATORY_PCT': 0.15,
    'TOTAL_PCT_TARGET': 1.0,
    'TOTAL_PCT_TOLERANCE': 0.1,
    'FLAVOR_WEIGHT': 0.8,
    'SIMPLICITY_WEIGHT': 0.2,
    'PENALTY_STRENGTH': 0.7,
    'PLOT_EVERY_N_GENS': 5
}

In [12]:
try:
    # In Lambda, this model would ideally be loaded from EFS, an S3 download at init, or a Lambda Layer.
    # For local testing, we load it from the mk3/model_output directory.
    predictor = joblib.load('model_output/model.joblib')
    ingredient_columns = predictor.feature_columns
    flavor_categories = predictor.flavor_categories
    optimal_thresholds = predictor.thresholds
    
    print(f"Production predictor loaded successfully!")
    print(f"Number of ingredients: {len(ingredient_columns)}")
    
except Exception as e:
    print(f"Error loading model: {e}")

ingredient_to_index = {ing: idx for idx, ing in enumerate(ingredient_columns)}

Production predictor loaded successfully!
Number of ingredients: 242


In [13]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

def normalize_individual(individual):
    total = sum(individual)
    if total > 0:
        scale_factor = GA_CONFIG['TOTAL_PCT_TARGET'] / total
        for i in range(len(individual)):
            individual[i] *= scale_factor
    return individual

def create_individual(mandatory_indices=None, mandatory_pcts=None):
    individual = [0.0] * len(ingredient_columns)
    
    if mandatory_indices:
        for i, idx in enumerate(mandatory_indices):
            if mandatory_pcts and i < len(mandatory_pcts):
                pct = mandatory_pcts[i]
            else:
                pct = random.uniform(GA_CONFIG['MIN_MANDATORY_PCT'], GA_CONFIG['MAX_INGREDIENT_PCT'])
            individual[idx] = pct
            
    num_optional = random.randint(0, 10)
    available_indices = [i for i in range(len(individual)) if i not in (mandatory_indices or [])]
    
    if available_indices and num_optional > 0:
        chosen_indices = random.sample(available_indices, min(num_optional, len(available_indices)))
        for idx in chosen_indices:
            individual[idx] = random.uniform(GA_CONFIG['MIN_INGREDIENT_PCT'], GA_CONFIG['MAX_INGREDIENT_PCT'])
            
    normalize_individual(individual)
    return creator.Individual(individual)

def cxBlend(ind1, ind2):
    for i in range(len(ind1)):
        if random.random() < 0.5:
            alpha = random.uniform(-0.5, 1.5)
            ind1[i] = alpha * ind1[i] + (1 - alpha) * ind2[i]
            ind2[i] = alpha * ind2[i] + (1 - alpha) * ind1[i]
            
    for ind in [ind1, ind2]:
        for i in range(len(ind)):
            if ind[i] < 0: ind[i] = 0
            elif ind[i] > GA_CONFIG['MAX_INGREDIENT_PCT']: ind[i] = GA_CONFIG['MAX_INGREDIENT_PCT']
            
    normalize_individual(ind1)
    normalize_individual(ind2)
    return ind1, ind2

def mutGaussian(individual):
    for i in range(len(individual)):
        if random.random() < 0.1:
            if individual[i] > 0:
                individual[i] += random.gauss(0, GA_CONFIG['MUTATION_STRENGTH'])
                individual[i] = max(0, min(GA_CONFIG['MAX_INGREDIENT_PCT'], individual[i]))
            elif random.random() < 0.05:
                individual[i] = random.uniform(GA_CONFIG['MIN_INGREDIENT_PCT'], GA_CONFIG['MAX_INGREDIENT_PCT'])
                
    if random.random() < 0.1:
        active_indices = [i for i, val in enumerate(individual) if val > GA_CONFIG['MIN_INGREDIENT_PCT']]
        if active_indices:
            idx = random.choice(active_indices)
            individual[idx] = 0.0
            
    normalize_individual(individual)
    return individual,

toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", cxBlend)
toolbox.register("mutate", mutGaussian)
toolbox.register("select", tools.selTournament, tournsize=GA_CONFIG['TOURNAMENT_SIZE'])

In [14]:
def evaluate_population(population, target_flavors, mandatory_indices=None):
    # 1. Stack all individuals into a 2D array
    X_batch = np.array(population)  # shape: (n_individuals, 242)
    
    # 2. Get predictions for all individuals simultaneously using the Scikit-Learn universal wrapper
    all_flavor_probs = predictor.predict_proba(X_batch) 
    
    n_individuals = len(population)
    fitnesses = []
    
    for i in range(n_individuals):
        flavor_fitness = 0
        total_weight = 0
        individual = population[i]
        
        # Calculate flavor score using pre-computed batch probabilities
        for flavor, weight in target_flavors.items():
            if flavor in flavor_categories:
                proba = all_flavor_probs[flavor][i]
                threshold = optimal_thresholds.get(flavor, 0.5)
                if proba < threshold:
                    proba *= GA_CONFIG['PENALTY_STRENGTH']  # Penalize if it fails to cross boundary
                flavor_fitness += proba * weight
                total_weight += weight
        
        if total_weight > 0:
            flavor_fitness /= total_weight
            
        # Simplicity Penalty
        active_ingredients = sum(1 for x in individual if x > GA_CONFIG['MIN_INGREDIENT_PCT'])
        simplicity_score = max(0, 8 - active_ingredients) / 8.0
        
        # Mandatory & Bounds Penalty
        mandatory_penalty = 0
        if mandatory_indices:
            for idx in mandatory_indices:
                if individual[idx] < GA_CONFIG['MIN_MANDATORY_PCT']:
                    mandatory_penalty += 0.2
        for x in individual:
            if x > GA_CONFIG['MAX_INGREDIENT_PCT']:
                mandatory_penalty += 0.2
                
        # Total Percentage Penalty
        total_pct = sum(individual)
        total_penalty = abs(total_pct - GA_CONFIG['TOTAL_PCT_TARGET']) / GA_CONFIG['TOTAL_PCT_TOLERANCE']
        total_penalty = min(total_penalty, 1.0)
        
        # Combine Fitness
        fitness = (GA_CONFIG['FLAVOR_WEIGHT'] * flavor_fitness + 
                   GA_CONFIG['SIMPLICITY_WEIGHT'] * simplicity_score - 
                   mandatory_penalty - total_penalty)
                   
        fitnesses.append((max(0, fitness),))
        
    return fitnesses

In [ ]:
def run_cocktail_ga(target_flavors, mandatory_ingredients=None, mandatory_percentages=None, verbose=True):
    print("\n" + "=" * 80)
    print("COCKTAIL GENETIC ALGORITHM OPTIMIZATION")
    print("=" * 80)
    
    mandatory_indices = []
    if mandatory_ingredients:
        for ing in mandatory_ingredients:
            if ing in ingredient_to_index:
                mandatory_indices.append(ingredient_to_index[ing])
            else:
                if verbose: print(f"Warning: Mandatory ingredient '{ing}' not found in ingredient list")
                
    def create_individual_with_mandatory():
        return create_individual(mandatory_indices, mandatory_percentages)
        
    toolbox.register("individual", create_individual_with_mandatory)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    # Initialize Population
    pop = toolbox.population(n=GA_CONFIG['POPULATION_SIZE'])
    
    # 🟢 Batch Evaluate Initial Population
    fitnesses_tuples = evaluate_population(pop, target_flavors, mandatory_indices)
    fitnesses = []
    for ind, fit in zip(pop, fitnesses_tuples):
        ind.fitness.values = fit
        fitnesses.append(fit[0])
        
    if verbose:
        print(f"\nTarget flavors: {target_flavors}")
        if mandatory_ingredients:
            print(f"Mandatory ingredients: {mandatory_ingredients}")
        print(f"Initial population: {len(pop)} individuals")
        print(f"Initial fitness - Min: {np.min(fitnesses):.3f}, "
              f"Avg: {np.mean(fitnesses):.3f}, Max: {np.max(fitnesses):.3f}")
              
    # Setup Statistics
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Setup Logbook & Hall Of Fame
    logbook = tools.Logbook()
    logbook.header = ["gen", "nevals"] + stats.fields
    hall_of_fame = tools.HallOfFame(GA_CONFIG['ELITISM_COUNT'])
    
    # Evolution Loop
    for gen in range(GA_CONFIG['GENERATIONS']):
        offspring = toolbox.select(pop, len(pop) - GA_CONFIG['ELITISM_COUNT'])
        offspring = list(map(toolbox.clone, offspring))
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < GA_CONFIG['CROSSOVER_PROB']:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
                
        for mutant in offspring:
            if random.random() < GA_CONFIG['MUTATION_PROB']:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                
        # Batch Evaluate Offspring
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        if invalid_ind:
            fitnesses_tuples = evaluate_population(invalid_ind, target_flavors, mandatory_indices)
            for ind, fit in zip(invalid_ind, fitnesses_tuples):
                ind.fitness.values = fit
                
        # Update Population & Hall of Fame
        pop = tools.selBest(pop, GA_CONFIG['ELITISM_COUNT']) + offspring
        hall_of_fame.update(pop)
        
        # Record & Print Generation Statistics
        record = stats.compile(pop)
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        
        plot_frequency = GA_CONFIG.get('PLOT_EVERY_N_GENS', 5)
        if verbose and gen % plot_frequency == 0:
            print(f"   Generation {gen:3d}: Avg fitness = {record['avg']:.3f}, "
                  f"Best = {record['max']:.3f}")

    # Finalize
    best_individual = tools.selBest(pop, 1)[0]
    
    if verbose:
        print(f"\n Optimization complete after {GA_CONFIG['GENERATIONS']} generations!")
        print(f"Best fitness: {best_individual.fitness.values[0]:.3f}")
        
    return best_individual, logbook, hall_of_fame

In [16]:
def parse_results(individual):
    """Format the raw recipe output into a clean JSON response"""
    recipe = []
    for idx, pct in enumerate(individual):
        if pct > GA_CONFIG['MIN_INGREDIENT_PCT']:
            recipe.append({
                "ingredient": ingredient_columns[idx],
                "percentage": round(pct * 100, 2)
            })
    recipe.sort(key=lambda x: x['percentage'], reverse=True)
    return recipe

def lambda_handler(event, context):
    """
    AWS Lambda entry point. 
    Expects API Gateway JSON payload:
    {
      "flavors": {"citrus": 0.8, "spicy": 0.2},
      "mandatory_ingredients": ["vodka_pct"]
    }
    """
    try:
        # 1. Parse API input
        body = json.loads(event.get('body', '{}')) if isinstance(event.get('body'), str) else event
        target_flavors = body.get('flavors', {'citrus': 1.0})
        mandatory_ingredients = body.get('mandatory_ingredients', [])
        
        start_time = time.time()
        
        # 2. Run highly optimized GA
        best_recipe_array, logbook, hall_of_fame = run_cocktail_ga(target_flavors, mandatory_ingredients)
        
        # 3. Format winning recipe
        final_recipe = parse_results(best_recipe_array)
        execution_time = round(time.time() - start_time, 2)
        
        # 4. Return via API Gateway
        return {
            'statusCode': 200,
            'headers': {'Content-Type': 'application/json'},
            'body': json.dumps({
                'message': 'Cocktail generated successfully!',
                'recipe': final_recipe,
                'ga_fitness_score': best_recipe_array.fitness.values[0],
                'execution_time_seconds': execution_time
            })
        }
        
    except Exception as e:
        return {
            'statusCode': 500,
            'body': json.dumps({'error': str(e)})
        }

In [17]:
if __name__ == '__main__':
    # Simulate an HTTP POST request from API Gateway
    mock_event = {
        "flavors": {"sweet": 0.6, "creamy": 0.4},
        "mandatory_ingredients": ["gin_pct"]
    }
    
    print("Testing Lambda Handler locally...")
    response = lambda_handler(mock_event, None)
    
    print("\nAPI Gateway Response:")
    print(json.dumps(json.loads(response['body']), indent=2))


Testing Lambda Handler locally...

COCKTAIL GENETIC ALGORITHM OPTIMIZATION

 Target flavors: {'sweet': 0.6, 'creamy': 0.4}
Mandatory ingredients: ['gin_pct']
Initial population: 1000 individuals
Initial fitness - Min: 0.000, Avg: 0.085, Max: 0.859
   Generation   0: Avg fitness = 0.152, Best = 0.859
   Generation   5: Avg fitness = 0.710, Best = 0.883
   Generation  10: Avg fitness = 0.808, Best = 0.905
   Generation  15: Avg fitness = 0.824, Best = 0.925
   Generation  20: Avg fitness = 0.841, Best = 0.925
   Generation  25: Avg fitness = 0.877, Best = 0.925
   Generation  30: Avg fitness = 0.873, Best = 0.925
   Generation  35: Avg fitness = 0.865, Best = 0.925
   Generation  40: Avg fitness = 0.859, Best = 0.925
   Generation  45: Avg fitness = 0.858, Best = 0.925
   Generation  50: Avg fitness = 0.858, Best = 0.925
   Generation  55: Avg fitness = 0.854, Best = 0.925
   Generation  60: Avg fitness = 0.852, Best = 0.925
   Generation  65: Avg fitness = 0.844, Best = 0.925
   Generat